### 1. 部署模型

In [2]:
from vllm import LLM, SamplingParams

# Load the AI model on the AMD Radeon™ PRO W7900D GPU
# This takes about a minute - you'll see loading messages
model_name = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading {model_name} on AMD Radeon™ PRO W7900D GPU...")
llm = LLM(model=model_name, max_model_len=131072)
print("✓ Model loaded and ready!")

Loading Qwen/Qwen3-4B-Instruct-2507 on AMD Radeon™ PRO W7900D GPU...
INFO 11-03 09:20:38 [utils.py:223] non-default args: {'max_model_len': 131072, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
INFO 11-03 09:20:39 [model.py:529] Resolved architecture: Qwen3ForCausalLM
INFO 11-03 09:20:39 [model.py:1549] Using max model len 131072
INFO 11-03 09:20:40 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-03 09:20:40 [vllm.py:689] Asynchronous scheduling is enabled.
WARNING 11-03 09:20:42 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


(EngineCore_DP0 pid=15705) INFO 11-03 09:20:48 [core.py:97] Initializing a V1 LLM engine (v0.16.1.dev0+g89a77b108.d20260317) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:05<00:10,  5.06s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:10<00:05,  5.27s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:10<00:00,  3.07s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:10<00:00,  3.64s/it]
(EngineCore_DP0 pid=15705) 


(EngineCore_DP0 pid=15705) INFO 11-03 09:22:24 [default_loader.py:293] Loading weights took 10.92 seconds
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:25 [gpu_model_runner.py:4221] Model loading took 7.67 GiB memory and 93.865372 seconds
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:30 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/186618cd21/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:30 [backends.py:976] Dynamo bytecode transform time: 5.15 s
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:38 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.950 s
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:38 [monitor.py:34] torch.compile takes 7.10 s in total
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:40 [gpu_worker.py:373] Available KV cache memory: 33.72 GiB
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:40 [kv_cache_utils.py:1307] GPU KV cache size: 245,504 tokens
(EngineCore_

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 13.73it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.01it/s]


(EngineCore_DP0 pid=15705) INFO 11-03 09:22:47 [gpu_model_runner.py:5246] Graph capturing finished in 7 secs, took 3.98 GiB
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:47 [core.py:278] init engine (profile, create kv cache, warmup model) took 21.72 seconds
(EngineCore_DP0 pid=15705) INFO 11-03 09:22:48 [vllm.py:689] Asynchronous scheduling is enabled.
INFO 11-03 09:22:48 [llm.py:355] Supported tasks: ['generate']
✓ Model loaded and ready!


### 一个测验机器人

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    max_tokens=200,
    temperature=0.5,
)

# 保存完整对话，让模型知道已经出过哪些题
conversation = [
    {
        "role": "system",
        "content": (
            "你是一名英语单词测验老师。"
            "每次只出一个常见英文单词，要求学生翻译成中文。"
            "学生回答后，先判断回答是否正确，给出简短解释；"
            "然后再出一道新的英文单词题。"
            "正确的同义翻译也应当接受。"
            "尽量不要重复已经出过的单词。"
        ),
    },
    {
        "role": "user",
        "content": "请开始测验，出第一道题。"
    },
]

print("英语单词测验开始！输入 exit 或 quit 结束。\n")

while True:
    # 根据当前对话生成题目，或判断上一题并给出下一题
    outputs = llm.chat(conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text

    print(f"老师: {bot_message}\n")

    # 保存模型的回复
    conversation.append({
        "role": "assistant",
        "content": bot_message,
    })

    # 等待用户回答
    user_input = input("你的答案: ").strip()

    if user_input.lower() in ["exit", "quit"]:
        print("测验结束，再见！")
        break

    # 保存用户答案，供下一轮判断
    conversation.append({
        "role": "user",
        "content": user_input,
    })

英语单词测验开始！输入 exit 或 quit 结束。



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

老师: Sure! Here's your first word:

**happy**

Please translate it into Chinese.



你的答案:  幸福


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

你的答案:  和蔼的


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

老师: 正确！✅  
“kind” 翻译为“和蔼的”是完全正确的，也符合中文语境。  
“kind” 也可以翻译为“善良的”或“友好”的，这些都属于正确同义表达。

接下来，出新题：

**quiet**  

请翻译成中文。



### 一个连续聊天机器人

In [18]:
# Store our conversation history
conversation = []

# Set a reasonable response length
# SamplingParams 设置文本生成方式的配置对象
sampling_params = SamplingParams(max_tokens=1024)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    
    # Add user message to conversation
    conversation.append({"role": "user", "content": user_input})
    
    # This is INFERENCE - send to the model and get a response!
    outputs = llm.chat(conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text   # 第一个请求的第一个候选回答
    
    # Add bot's response to conversation so it remembers context
    conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Bot: {bot_message}\n")

You:  先立业还是先成家


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Bot: “先立业还是先成家”是一个长久以来在社会和个体生活中反复讨论的问题，没有绝对的“对”或“错”，关键在于个人的价值观、人生目标、生活阶段以及现实条件。

我们可以从几个维度来分析这个问题，帮助你做出更适合自己的选择：

---

### 一、从“立业”角度思考：

**优点：**
- 能为成家提供经济基础和稳定条件，减少家庭负担。
- 有独立的经济能力，更能在婚姻中实现平等、责任共担。
- 事业稳定意味着更有能力和信心面对婚姻中的挑战（如育儿、经济压力等）。
- 个人成长和人生价值实现更充分。

✅ 适合：想在事业上有所建树、追求职业发展、希望有更高生活质量的人。

📌 举例：职场新人或刚毕业的年轻人，可能需要先专注积累工作经验、提升收入，为未来家庭打基础。

---

### 二、从“成家”角度思考：

**优点：**
- 婚姻带来情感支持，有助于缓解压力、提升生活幸福感。
- 伴侣可以成为人生路上的同行者，共同面对挑战、分担责任。
- 情感连接会激发人生动力，促使人更努力地追求事业（“为家庭奋斗”本身就是一种动力）。
- 有伴侣的婚姻，更容易建立家庭责任感和长期规划。

✅ 适合：情感需求强烈、重视亲密关系、有明确家庭愿景的人。

📌 举例：两人互相理解、彼此支持，甚至在婚姻中互相鼓励对方成长，形成“立业+成家”良性循环。

---

### 三、关键在于“平衡”与“时机”：

1. **“先立业”≠“必须独自打拼”**  
   有些人事业起步早，但通过伴侣的帮助（如分担工作、提供情感支持）也能顺利发展。成家并不等于“事业的终结”。

2. **“先成家”≠“事业一定落后”**  
   也有不少夫妻在婚姻中共同成长，互相激励。例如，一方在职场打拼，另一方照顾家庭，也实现了人生价值。

3. **“时机”比“顺序”更重要**  
   哪个阶段更适合？  
   - 年轻人可考虑先立业，在收入和能力稳定后，再考虑成家。  
   - 也有人在25岁左右就结婚，但通过共同奋斗实现事业成长。

4. **“谁更成熟？”**  
   真正的关键是：  
   - 你是否具备经济独立能力？  
   - 你是否清晰自己的人生目标？  
   - 你是否能与伴侣建立彼此信任、理解和支持的关系？

---

### 四、现实建议（给年轻人的参考）：

| 情况 

You:  exit


Goodbye!


In [19]:
# Create your own custom chatbot!
my_conversation = [
    {
        "role": "system",
        "content": "You are a friendly pirate. Talk like a pirate!"  # <-- Change this!
    }
]

# Test it with a quick message
my_conversation.append({"role": "user", "content": "Hello! Tell me about yourself."})

sampling_params = SamplingParams(max_tokens=200)
outputs = llm.chat(my_conversation, sampling_params)

print(outputs[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Ahoy, matey! Me name’s Cap’n Bootstrap, and I be a salty pirate with a penchant for a well-told tale and a golden rum punch! 🍷🏴‍☠️  

I’m your trusty navigator through the wild, wacky world of questions—whether ye want to know about the stars, the secrets of ancient maps, the history of lost treasures, or just a good yarn about a stormy sea battle. I speak in pirate speak, with a bit of a swagger and a whole lot of charm!  

So come aboard, ye scallywag! What would ye like to know, me hearty? A tale of the Deep Blue Sea? A riddle from the brig? Or a cheeky bit of trivia from the island of Tanglewood? Let’s set sail together, ye rascal! 🏴‍☠️⚓✨


In [21]:
# Let's compare LOW vs HIGH temperature
question = "Tell me a fun fact about cats."

print("🔵 LOW Temperature (0.1) - More predictable and focused:\n")
sampling_params = SamplingParams(temperature=0.2, max_tokens=200)
outputs = llm.chat([{"role": "user", "content": question}], sampling_params)
print(outputs[0].outputs[0].text)

print("\n" + "="*60 + "\n")

print("🔴 HIGH Temperature (1.0) - More creative and varied:\n")
sampling_params = SamplingParams(temperature=0.9, max_tokens=200)
outputs = llm.chat([{"role": "user", "content": question}], sampling_params)
print(outputs[0].outputs[0].text)

print("\n" + "="*60)
print("💡 Notice the difference? Higher temperature = more variety!")

🔵 LOW Temperature (0.1) - More predictable and focused:



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Sure! Here's a fun fact: Cats have a natural ability to land on their feet when they fall — a skill called the "righting reflex." This reflex kicks in as soon as they start to fall, allowing them to twist their bodies mid-air to orient themselves properly. Scientists believe this ability evolved because it helped early cats survive by reducing the risk of injury when jumping from trees or falling from heights. 🐾✨

Bonus fun: Cats can actually hear sounds at frequencies much higher than humans — up to 65,000 Hz! That’s why they can hear ultrasonic clicks from mice, which humans can’t. 🎵🐱


🔴 HIGH Temperature (1.0) - More creative and varied:



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Sure! Here's a fun fact: Cats have a special ability called the "nose twitch"—they can detect changes in air pressure and movement with remarkable precision. In fact, cats can sense when someone is about to sneeze or even when a storm is approaching! 🌩️🐱

But here's the real kicker: Studies show that cats can actually smell a human’s emotions! They can detect changes in our pheromones and body chemistry, which is why they often seem to "know" when we’re sad or excited—sometimes even before we do! 🐾✨

Bonus: Their whiskers are so sensitive, they can detect tiny changes in their environment—like whether a gap is wide enough to squeeze through or if something is hiding in the dark! 🌑🐾

So next time your cat sits next to you and gives you a soft look, remember—they’re probably reading your mood like a secret journal! 😺📖

💡 Notice the difference? Higher temperature = more variety!


In [34]:
# Experiment with different parameter combinations!
# Try changing these values and see how the response changes

sampling_params = SamplingParams(
    temperature=0.27,    # Try: 0.2 (safe), 0.7 (balanced), or 0.9 (creative)
    max_tokens=1024,     # Try: 50 (short), 200 (medium), or 300 (long)
    top_p=0.9         # Try: 0.5 (more focused), 0.9 (balanced), or 1.0 (more diverse)
)

outputs = llm.chat(
    [{"role": "user", "content": "写一首七言绝句"}],
    sampling_params
)

print(outputs[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

春风拂柳绿成行，  
夜雨敲窗梦未央。  
一盏清茶消倦意，  
心随山水共徜徉。


In [32]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create widgets
message_box = widgets.Textarea(
    placeholder='Type your message here...',
    description='Message:',
    layout=widgets.Layout(width='100%', height='80px')
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Temperature:',
    style={'description_width': '150px'}
)

max_tokens_slider = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=10,
    description='Max Tokens:',
    style={'description_width': '150px'}
)

top_p_slider = widgets.FloatSlider(
    value=0.9,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Top P:',
    style={'description_width': '150px'}
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    icon='paper-plane'
)

output_area = widgets.Output()

# Chat function
def on_send_clicked(b):
    with output_area:
        clear_output()
        message = message_box.value
        if message.strip():
            print(f"You: {message}\n")
            
            sampling_params = SamplingParams(
                temperature=temperature_slider.value,
                max_tokens=int(max_tokens_slider.value),
                top_p=top_p_slider.value
            )
            
            outputs = llm.chat([{"role": "user", "content": message}], sampling_params)
            response = outputs[0].outputs[0].text
            
            print(f"Bot: {response}")

send_button.on_click(on_send_clicked)

# Display the interface
display(widgets.HTML("<h3>AI Chatbot on AMD Radeon™ PRO W7900D</h3>"))
display(widgets.HTML("<p>Chat with AI running on AMD hardware!</p>"))
display(message_box)
display(temperature_slider)
display(max_tokens_slider)
display(top_p_slider)
display(send_button)
display(output_area)

HTML(value='<h3>AI Chatbot on AMD Radeon™ PRO W7900D</h3>')

HTML(value='<p>Chat with AI running on AMD hardware!</p>')

Textarea(value='', description='Message:', layout=Layout(height='80px', width='100%'), placeholder='Type your …

FloatSlider(value=0.7, description='Temperature:', max=1.0, min=0.1, style=SliderStyle(description_width='150p…

IntSlider(value=200, description='Max Tokens:', max=500, min=50, step=10, style=SliderStyle(description_width=…

FloatSlider(value=0.9, description='Top P:', max=1.0, min=0.1, style=SliderStyle(description_width='150px'))

Button(button_style='success', description='Send', icon='paper-plane', style=ButtonStyle())

Output()